In [3]:
import os 
import requests
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY") #get API key from env file

BASE_URL = "https://www.alphavantage.co/query" #base url for api

def fetch_stock_data(symbol):
    
    """
    Fetch daily stock data from Alpha API
    """

    
    params = {
        "function" : "TIME_SERIES_DAILY",
        "symbol": symbol,
        "apikey" : API_KEY 
    }

    response = requests.get(BASE_URL, params=params)
    
    if response.status_code != 200 :
        raise Exception("API request dailed")

   
    data = response.json() 
    
    if "Error Message" in data:
        raise Exception("Invalid API call")

    if "Note" in data:
        raise Exception("API rate limit exceeded")

    return data 

def parse_stock_data(data) :
    """
    Extract time serise data from API response
    """

    if "Time Series (Daily)" not in data:
        raise Exception("Time Series data not found in API response")
    
    return data["Time Series (Daily)"]

import pandas as pd

def convert_to_df(df) :
    return pd.DataFrame(df).T


def time_serise_analysis_dataset (df) :
    df = df.copy() #copy this dataset so that for delink the referance 
    df = df.drop_duplicates() #drop the duplicate value
    df = df.dropna()  #drop the missing value KAN-39
    df = df.rename(columns={                            
            "1. open" :"open",
            "2. high" :"high" ,  
            "3. low"  :"low" , 
            "4. close" :"close"  ,
            "5. volume" : "volume" }) #rename the column
    df = df.astype({"open":float,"high":float,"low":float,"close":float,"volume":int}).round(2) #reasign the data type and round it by 2

    df.index = pd.to_datetime(df.index) #set index value as date time
    
    df = df.sort_index(ascending=False) # sorted the index

    df.reset_index(inplace=True)
    df.rename(columns={"index":"Date"},inplace=True) #change the index name as date 
    df = df.set_index("Date") #set this cloumn as date time 
    return df
def time_serise_feature_engineering(df) :
    
    df["close_norm"] = (df["close"] /  df["close"].max()).astype(float).round(2)  #normalized this stock value KAN-40 


    df["retuens"] = (df["close"].pct_change().fillna(0)).astype(float).round(4) #calculatethe daily return KAN-41

        

    df["pct_change"] = (df["close"].pct_change().fillna(0) * 100).astype(float).round(2) #calculate the percentage chnage
    
    return df


def compute_all_statistics(df) :
    stats = {}

    # core return stats
    stats["mean"] = round(float(df["retuens"].mean()),6)
    stats["median"] = round(float(df["retuens"].median()),6)
    stats["std_dev"] = round(float(df["retuens"].std()),6)
    stats["variance"] =  round(float(df["retuens"].var()),6)

    # Extremes
    stats["min"] =  round(float(df["retuens"].min()),6)
    stats["max"] =  round(float(df["retuens"].max()),6)

    #Percentiles
    stats["percentiles"] = {
        "25" :round(float(df["retuens"].quantile(0.25)),6),
        "50" :round(float(df["retuens"].quantile(0.50)),6),
        "75" :round(float(df["retuens"].quantile(0.75)),6),
        "5"  :round(float(df["retuens"].quantile(0.05)),6),
        "95" :round(float(df["retuens"].quantile(0.95)),6),

    }

    return stats

In [4]:
def get_stock_analysis(symbol) :

     data = fetch_stock_data(symbol)  #get the overall stock data
     time_series = parse_stock_data(data) #get the time series data
     time_series_data_frame = convert_to_df(time_series) #convert to the dataframe
     time_series_data_frame = time_serise_analysis_dataset(time_series_data_frame)
     time_series_data_frame = time_serise_feature_engineering(time_series_data_frame)  # create analysis data
     
     stats = compute_all_statistics(time_series_data_frame)  #Extract the stats out of it 

     return {
          "symbol" : symbol ,
          "stats" : stats,
          "Data": time_series_data_frame , #return only the first 10 records for now
          
     }




In [5]:
data = get_stock_analysis("AAPL")
symbol = data["symbol"]
final_output = data["Data"]

final_output.keys()



Index(['open', 'high', 'low', 'close', 'volume', 'close_norm', 'retuens',
       'pct_change'],
      dtype='str')

In [6]:
final_output.head()

,open,high,low,close,volume,close_norm,retuens,pct_change
Date,,,,,,,,
2026-03-27,253.90,255.49,248.07,248.80,47899998,0.87,0.0000,0.00
2026-03-26,252.12,257.00,250.77,252.89,41796650,0.88,0.0164,1.64
2026-03-25,254.10,255.00,251.60,252.62,28476668,0.88,-0.0011,-0.11
2026-03-24,250.35,254.82,249.55,251.64,45152288,0.88,-0.0039,-0.39
2026-03-23,253.97,254.60,250.28,251.49,40546109,0.88,-0.0006,-0.06


In [7]:
# latest_date = "2026-03-23"

In [8]:
# new_df = final_output[final_output.index > latest_date]

In [9]:
# type(new_df.index[0])

In [10]:
records= []

for date, row in final_output.iterrows() :
    records.append({
        "date": date.strftime("%Y-%m-%d"),
        "open" : float(row["open"])
    })

        
    


In [11]:
records

[{'date': '2026-03-27', 'open': 253.9},
 {'date': '2026-03-26', 'open': 252.12},
 {'date': '2026-03-25', 'open': 254.1},
 {'date': '2026-03-24', 'open': 250.35},
 {'date': '2026-03-23', 'open': 253.97},
 {'date': '2026-03-20', 'open': 247.98},
 {'date': '2026-03-19', 'open': 249.4},
 {'date': '2026-03-18', 'open': 252.62},
 {'date': '2026-03-17', 'open': 252.96},
 {'date': '2026-03-16', 'open': 252.1},
 {'date': '2026-03-13', 'open': 255.48},
 {'date': '2026-03-12', 'open': 258.66},
 {'date': '2026-03-11', 'open': 261.09},
 {'date': '2026-03-10', 'open': 257.64},
 {'date': '2026-03-09', 'open': 255.69},
 {'date': '2026-03-06', 'open': 258.63},
 {'date': '2026-03-05', 'open': 260.79},
 {'date': '2026-03-04', 'open': 264.65},
 {'date': '2026-03-03', 'open': 263.48},
 {'date': '2026-03-02', 'open': 262.41},
 {'date': '2026-02-27', 'open': 272.81},
 {'date': '2026-02-26', 'open': 274.94},
 {'date': '2026-02-25', 'open': 271.78},
 {'date': '2026-02-24', 'open': 267.86},
 {'date': '2026-02-2

In [12]:
final_output

,open,high,low,close,volume,close_norm,retuens,pct_change
Date,,,,,,,,
2026-03-27,253.90,255.49,248.07,248.80,47899998,0.87,0.0000,0.00
2026-03-26,252.12,257.00,250.77,252.89,41796650,0.88,0.0164,1.64
2026-03-25,254.10,255.00,251.60,252.62,28476668,0.88,-0.0011,-0.11
2026-03-24,250.35,254.82,249.55,251.64,45152288,0.88,-0.0039,-0.39
2026-03-23,253.97,254.60,250.28,251.49,40546109,0.88,-0.0006,-0.06
...,...,...,...,...,...,...,...,...
2025-11-07,269.80,272.29,266.77,268.47,48227365,0.94,-0.0036,-0.36
2025-11-06,267.89,273.40,267.89,269.77,51204045,0.94,0.0048,0.48
2025-11-05,268.61,271.70,266.93,270.14,42586288,0.94,0.0014,0.14


In [13]:
final_output.head()

,open,high,low,close,volume,close_norm,retuens,pct_change
Date,,,,,,,,
2026-03-27,253.90,255.49,248.07,248.80,47899998,0.87,0.0000,0.00
2026-03-26,252.12,257.00,250.77,252.89,41796650,0.88,0.0164,1.64
2026-03-25,254.10,255.00,251.60,252.62,28476668,0.88,-0.0011,-0.11
2026-03-24,250.35,254.82,249.55,251.64,45152288,0.88,-0.0039,-0.39
2026-03-23,253.97,254.60,250.28,251.49,40546109,0.88,-0.0006,-0.06


In [14]:
final_output.index = final_output.index.strftime("%Y-%m-%d")

In [15]:
latest_date = "2026-03-23"  


In [21]:
print(final_output[final_output.index > latest_date])

              open    high     low   close    volume  close_norm  retuens  \
Date                                                                        
2026-03-27  253.90  255.49  248.07  248.80  47899998        0.87   0.0000   
2026-03-26  252.12  257.00  250.77  252.89  41796650        0.88   0.0164   
2026-03-25  254.10  255.00  251.60  252.62  28476668        0.88  -0.0011   
2026-03-24  250.35  254.82  249.55  251.64  45152288        0.88  -0.0039   

            pct_change  
Date                    
2026-03-27        0.00  
2026-03-26        1.64  
2026-03-25       -0.11  
2026-03-24       -0.39  


In [34]:
records = []
for Date, row in final_output.iterrows() :
    records.append({
        "Date" : Date,
        "Open":float(row["open"]),
        "Close" : float(row["open"]),
    })

In [35]:
records

[{'Date': '2026-03-27', 'Open': 253.9, 'Close': 253.9},
 {'Date': '2026-03-26', 'Open': 252.12, 'Close': 252.12},
 {'Date': '2026-03-25', 'Open': 254.1, 'Close': 254.1},
 {'Date': '2026-03-24', 'Open': 250.35, 'Close': 250.35},
 {'Date': '2026-03-23', 'Open': 253.97, 'Close': 253.97},
 {'Date': '2026-03-20', 'Open': 247.98, 'Close': 247.98},
 {'Date': '2026-03-19', 'Open': 249.4, 'Close': 249.4},
 {'Date': '2026-03-18', 'Open': 252.62, 'Close': 252.62},
 {'Date': '2026-03-17', 'Open': 252.96, 'Close': 252.96},
 {'Date': '2026-03-16', 'Open': 252.1, 'Close': 252.1},
 {'Date': '2026-03-13', 'Open': 255.48, 'Close': 255.48},
 {'Date': '2026-03-12', 'Open': 258.66, 'Close': 258.66},
 {'Date': '2026-03-11', 'Open': 261.09, 'Close': 261.09},
 {'Date': '2026-03-10', 'Open': 257.64, 'Close': 257.64},
 {'Date': '2026-03-09', 'Open': 255.69, 'Close': 255.69},
 {'Date': '2026-03-06', 'Open': 258.63, 'Close': 258.63},
 {'Date': '2026-03-05', 'Open': 260.79, 'Close': 260.79},
 {'Date': '2026-03-04'

In [36]:
final_output

,open,high,low,close,volume,close_norm,retuens,pct_change
Date,,,,,,,,
2026-03-27,253.90,255.49,248.07,248.80,47899998,0.87,0.0000,0.00
2026-03-26,252.12,257.00,250.77,252.89,41796650,0.88,0.0164,1.64
2026-03-25,254.10,255.00,251.60,252.62,28476668,0.88,-0.0011,-0.11
2026-03-24,250.35,254.82,249.55,251.64,45152288,0.88,-0.0039,-0.39
2026-03-23,253.97,254.60,250.28,251.49,40546109,0.88,-0.0006,-0.06
...,...,...,...,...,...,...,...,...
2025-11-07,269.80,272.29,266.77,268.47,48227365,0.94,-0.0036,-0.36
2025-11-06,267.89,273.40,267.89,269.77,51204045,0.94,0.0048,0.48
2025-11-05,268.61,271.70,266.93,270.14,42586288,0.94,0.0014,0.14
